In [ ]:
import random
import sys
from functools import partial
from pathlib import Path

import numpy as np
import torch
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION

if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent for path in INPUT_ROOT.rglob("annotations_coco.json")
        if path.parent.name == "train"
    })

    if len(data_roots) != 1:
        raise FileNotFoundError("Set DIRECTORY to the attached competition data folder.")

    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "model_builders.py",
    "torch_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("experiment_runner.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file() for filename in required_scripts)
})

scripts_dir = script_dirs[0]
scripts_parent = str(scripts_dir.parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.model_builders import build_ssd_model
from scripts.experiment_runner import run_torch_cv
from scripts.experiment_results import model_training_results

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=4.95s)
creating index...
index created!


In [ ]:
cv_fold_loaders = {}

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    train_dataset = DetectionDataset(frames=training_frames, validation=False, **kwargs_dataset)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "train": DataLoader(train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_data
        ),
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(train_dataset)} training frames, {len(valid_dataset)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


## SSD300 VGG16 baseline model - positive fraction changed

In [ ]:
positive_fraction_histories, positive_fraction_results = run_torch_cv(cv_fold_loaders=cv_fold_loaders,
    model_method=partial(build_ssd_model,  positive_fraction=0.5), output_dir=PROJECT_ROOT / "outputs" / "ssd_positive_fraction_0.5",
    annotations_path=INFO_TRAIN, device=device, seed=SEED, epochs=EPOCHS)

loading annotations into memory...
Done (t=5.01s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:00<00:00, 220MB/s]  


Loading and preparing results...
DONE (t=5.86s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=354.73s).
Accumulating evaluation results...
DONE (t=13.38s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.052
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.017
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.020
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.057
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.005
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.023
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.038
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets

In [2]:
positive_fraction_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="ssd_positive_fraction_0.5",
                                                        cv_folds=range(3))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.847178,0.0001,0.021902,0.051726,0.002038
1,2,3.985788,0.0001,0.022450,0.049141,0.002472
2,3,3.719420,0.0001,0.023392,0.050009,0.002834
3,4,3.491223,0.0001,0.030561,0.060355,0.002595
4,5,3.332864,0.0001,0.028817,0.057544,0.002736



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.078059,0.0001,0.024069,0.057830,0.001955
1,2,4.126461,0.0001,0.026207,0.058709,0.002217
2,3,3.763158,0.0001,0.038279,0.081199,0.002150
3,4,3.521509,0.0001,0.040653,0.082961,0.002325
4,5,3.348580,0.0001,0.040778,0.083547,0.003209



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.891161,0.0001,0.018106,0.044935,0.002812
1,2,4.080400,0.0001,0.020389,0.049179,0.002767
2,3,3.748428,0.0001,0.021652,0.050369,0.002678
3,4,3.522751,0.0001,0.023359,0.056771,0.002539
4,5,3.364603,0.0001,0.026028,0.059766,0.003566



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,4,3.491223,0.0001,0.030561,0.060355,0.002595
1,1,5,3.348580,0.0001,0.040778,0.083547,0.003209
2,2,5,3.364603,0.0001,0.026028,0.059766,0.003566



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.032456,0.067889,0.003123
std,0.007555,0.013563,0.000491
